In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("techsash/waste-classification-data")

print("Path to dataset files:", path)

In [ ]:
train_dir = os.path.join(path, "DATASET", "TRAIN")
train_img_o = os.path.join(path, "DATASET", "TRAIN", 'O')
train_img_r = os.path.join(path, "DATASET", "TRAIN", 'R')
test_dir = os.path.join(path, "DATASET", "TEST")

In [ ]:
def show_img(temp):
  data_dir = Path(temp)
  images = sorted(list(map(str, list(data_dir.glob("*.jpg")))))
  plt.figure(figsize=(24,16))
  for i in range(1, 51):
    img = Image.open(images[i])
    plt.subplot(5,10,i)
    plt.imshow(img)
    plt.axis('off')

In [ ]:
show_img(train_img_o)

In [ ]:
show_img(train_img_r)

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest",
)

test_gen = ImageDataGenerator(
    rescale=1./255,
)

In [ ]:
train_data = train_gen.flow_from_directory(
    train_dir,
    target_size = (128, 128),
    batch_size = 16,
    class_mode="categorical",
)

test_data = test_gen.flow_from_directory(
    test_dir,
    target_size = (128, 128),
    batch_size = 16,
    class_mode="categorical",
)

In [ ]:
num_class = train_data.num_classes

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
model = models.Sequential([
    layers.Input(shape=(128,128,3)),

    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPool2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPool2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPool2D((2,2)),

    layers.Conv2D(256, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPool2D((2,2)),

    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(num_class, activation="softmax"),
])

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

In [ ]:
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    ModelCheckpoint("best_waste_cnn.h5", monitor="val_accuracy", save_best_only=True),
]

history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=30,
    callbacks=callbacks,
)

test_loss, test_acc = model.evaluate(test_data)
print(f"Final validation accuracy: {test_acc:.4f}")

model.save("waste_cnn_final.h5")